In [4]:
import pandas as pd

job_postings = pd.read_csv(r"C:\JOB_market_analysis\job_postings_fact.csv", parse_dates=['job_posted_date'])
company_dim  = pd.read_csv(r"C:\JOB_market_analysis\company_dim.csv")
skills_dim   = pd.read_csv(r"C:\JOB_market_analysis\skills_dim.csv")
skills_job   = pd.read_csv(r"C:\JOB_market_analysis\skills_job_dim.csv")

print(job_postings.shape, company_dim.shape, skills_dim.shape, skills_job.shape)

C:\Users\Manas\AppData\Local\Temp\ipykernel_4728\999209061.py:3: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  job_postings = pd.read_csv(r"C:\JOB_market_analysis\job_postings_fact.csv", parse_dates=['job_posted_date'])


(787686, 16) (140033, 5) (259, 3) (3669604, 2)


In [5]:
job_postings.head(10)

,job_id,company_id,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg
0,0,0,Data Analyst,Marketing Data Analyst,Anywhere,via LinkedIn,Full-time,True,Serbia,2023-09-25 17:46:00,False,False,Serbia,NaN,NaN,NaN
1,55,1,Cloud Engineer,Storage and Virtualization Engineer,"Kuwait City, Kuwait",via Trabajo.org,Full-time,False,Kuwait,2023-07-30 17:49:00,True,False,Kuwait,NaN,NaN,NaN
2,66,2,Data Analyst,Data Analyst et Scientist F/H,"Paris, France",via Emplois Trabajo.org,Full-time,False,France,2023-07-28 17:28:00,False,False,France,NaN,NaN,NaN
3,76,3,Data Engineer,Data Engineer,"Denver, CO",via LinkedIn,Contractor,False,"Illinois, United States",2023-04-03 17:14:00,False,False,United States,hour,NaN,70.0
4,81,4,Data Engineer,Data Engineer,Anywhere,via LinkedIn,Contractor,True,Canada,2023-03-25 17:25:00,False,False,Canada,NaN,NaN,NaN
5,105,5,Data Analyst,Data Analyst,"Bangkok, Thailand",via Jobtopgun.com,Full-time,False,Thailand,2023-01-29 17:16:00,False,False,Thailand,NaN,NaN,NaN
6,106,6,Data Engineer,Data Lead Engineer (with strong Python) - Remo...,Anywhere,via Jobgether,Full-time,True,Nicaragua,2023-08-13 17:37:00,False,False,Nicaragua,NaN,NaN,NaN
7,116,7,Senior Data Engineer,Senior Data Engineer,"New York, NY",via Melga,Full-time,False,"Florida, United States",2023-02-19 17:11:00,False,True,United States,NaN,NaN,NaN
8,122,8,Data Analyst,Full Time Data Analyst,"Chicago, IL",via Trabajo.org,Full-time,False,"Illinois, United States",2023-10-19 17:01:00,False,True,United States,NaN,NaN,NaN
9,134,9,Data Engineer,Data Engineer Remote / Telecommute Jobs,Anywhere,via Clearance Jobs,Full-time,True,Georgia,2023-04-27 17:38:00,True,False,United States,NaN,NaN,NaN


In [ ]:


import pandas as pd
import numpy as np
import re
import os

# ----------------------------------------------------------------------
# 0. CONFIG — change these two lines to match your machine
# ----------------------------------------------------------------------
RAW_DIR = r"C:\JOB_market_analysis"          # <-- your raw CSV folder
OUT_DIR = os.path.join(RAW_DIR, "cleaned")   # <-- cleaned output folder
os.makedirs(OUT_DIR, exist_ok=True)

HOURS_PER_YEAR = 2080  # 40 hrs/week x 52 weeks — used for hourly->yearly salary conversion


def log(msg):
    print(f"\n{'='*70}\n{msg}\n{'='*70}")


# ----------------------------------------------------------------------
# 1. LOAD
# ----------------------------------------------------------------------
def load_table(filename, **kwargs):
    path = os.path.join(RAW_DIR, filename)
    if not os.path.exists(path):
        print(f"  [SKIP] {filename} not found at {path}")
        return None
    df = pd.read_csv(path, **kwargs)
    print(f"  [OK] {filename}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df


log("STEP 1: LOADING RAW TABLES")
job_postings = load_table("job_postings_fact.csv", parse_dates=["job_posted_date"])
company_dim  = load_table("company_dim.csv")
skills_dim   = load_table("skills_dim.csv")
skills_job   = load_table("skills_job_dim.csv")


# ----------------------------------------------------------------------
# 2. CLEAN company_dim
# ----------------------------------------------------------------------
def clean_company_dim(df):
    log("STEP 2: CLEANING company_dim")
    df = df.copy()
    before = len(df)

    # Drop rows with no company name at all — a nameless company_id is unusable
    df = df.dropna(subset=["name"])
    print(f"  Dropped {before - len(df)} rows with null company name")

    # Trim whitespace on text fields
    for col in ["name", "link", "link_google"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace({"nan": np.nan})

    # thumbnail is a pure display/UI field with 42% nulls and zero analytical
    # value — drop it rather than impute a fake image URL
    if "thumbnail" in df.columns:
        df = df.drop(columns=["thumbnail"])
        print("  Dropped 'thumbnail' column (no analytical value)")

    # link (company website) is 61% missing — this is legitimate missingness
    # (many postings just don't have it), NOT something to impute. Instead,
    # add a boolean flag so downstream analysis can account for it if needed.
    if "link" in df.columns:
        df["has_website"] = df["link"].notna()

    # Flag duplicate names (different company_id, same name) — keep them,
    # just make it visible for QA rather than silently merging companies
    dup_mask = df["name"].str.lower().duplicated(keep=False)
    df["is_duplicate_name"] = dup_mask
    print(f"  Flagged {dup_mask.sum()} rows sharing a duplicate company name")

    print(f"  Final shape: {df.shape}")
    return df


# ----------------------------------------------------------------------
# 3. CLEAN skills_dim
# ----------------------------------------------------------------------
def clean_skills_dim(df):
    log("STEP 3: CLEANING skills_dim")
    df = df.copy()

    df["skills"] = df["skills"].astype(str).str.strip().str.lower()
    df["type"] = df["type"].astype(str).str.strip().str.lower()

    dupes = df.duplicated(subset=["skills"]).sum()
    if dupes:
        print(f"  Found {dupes} duplicate skill names — dropping duplicates, keeping first")
        df = df.drop_duplicates(subset=["skills"], keep="first")

    # A manual "modern vs core" tag — genuinely useful for the "high-value
    # skills" story instead of just raw frequency counts
    emerging_skills = {
        "dbt", "airflow", "snowflake", "databricks", "kafka", "dagster",
        "looker", "power bi", "dax", "pyspark", "spark"
    }
    df["skill_era"] = np.where(df["skills"].isin(emerging_skills), "emerging", "core")

    print(f"  Final shape: {df.shape}")
    return df


# ----------------------------------------------------------------------
# 4. CLEAN job_postings_fact
# ----------------------------------------------------------------------
def clean_job_postings(df):
    log("STEP 4: CLEANING job_postings_fact")
    df = df.copy()
    before = len(df)

    # --- de-dup on primary key ---
    if "job_id" in df.columns:
        dupes = df.duplicated(subset=["job_id"]).sum()
        df = df.drop_duplicates(subset=["job_id"])
        print(f"  Dropped {dupes} duplicate job_id rows")

    # --- salary standardization: yearly + hourly -> one comparable column ---
    # Do NOT average the two scales together blindly. Convert hourly to an
    # annualized figure, but keep an explicit flag of which source was used
    # so the conversion is auditable, not hidden.
    df["salary_standardized"] = df.get("salary_year_avg")

    has_hourly_only = df["salary_standardized"].isna() & df.get("salary_hour_avg").notna()
    df.loc[has_hourly_only, "salary_standardized"] = (
        df.loc[has_hourly_only, "salary_hour_avg"] * HOURS_PER_YEAR
    )

    df["salary_type"] = "missing"
    df.loc[df["salary_year_avg"].notna(), "salary_type"] = "yearly"
    df.loc[has_hourly_only, "salary_type"] = "hourly_converted"

    print("  Salary type breakdown:")
    print(df["salary_type"].value_counts().to_string())

    # --- outlier flagging (IQR method) — flag, don't silently delete ---
    valid_salary = df["salary_standardized"].dropna()
    q1, q3 = valid_salary.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df["salary_outlier"] = (df["salary_standardized"] < lower) | (df["salary_standardized"] > upper)
    print(f"  Flagged {df['salary_outlier'].sum()} salary outliers (IQR bounds: {lower:,.0f} - {upper:,.0f})")

    # --- text cleanup ---
    for col in ["job_title", "job_title_short", "job_location", "job_country",
                "job_via", "job_schedule_type"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    # --- seniority level, parsed from title ---
    def get_seniority(title):
        if not isinstance(title, str):
            return "Unspecified"
        t = title.lower()
        if any(w in t for w in ["principal", "staff", "head of", "director"]):
            return "Principal/Director"
        if any(w in t for w in ["senior", "sr.", "sr "]):
            return "Senior"
        if any(w in t for w in ["lead"]):
            return "Lead"
        if any(w in t for w in ["junior", "jr.", "jr ", "entry"]):
            return "Junior/Entry"
        return "Mid/Unspecified"

    title_col = "job_title" if "job_title" in df.columns else "job_title_short"
    if title_col in df.columns:
        df["seniority_level"] = df[title_col].apply(get_seniority)
        print("  Seniority breakdown:")
        print(df["seniority_level"].value_counts().to_string())

    # --- remote flag cleanup ---
    if "job_work_from_home" in df.columns:
        df["is_remote"] = df["job_work_from_home"].fillna(False).astype(bool)
    elif "job_location" in df.columns:
        df["is_remote"] = df["job_location"].str.lower().eq("anywhere")

    # --- date parts for trend analysis ---
    if "job_posted_date" in df.columns:
        df["posted_year"] = df["job_posted_date"].dt.year
        df["posted_month"] = df["job_posted_date"].dt.month
        df["posted_quarter"] = df["job_posted_date"].dt.quarter

    # --- degree / benefits flags coerced to real booleans ---
    for col in ["job_no_degree_mention", "job_health_insurance"]:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)

    print(f"  Rows before: {before:,} -> after: {len(df):,}")
    return df


# ----------------------------------------------------------------------
# 5. CLEAN skills_job_dim (bridge table) + referential integrity checks
# ----------------------------------------------------------------------
def clean_skills_job(df, job_postings, skills_dim):
    log("STEP 5: CLEANING skills_job_dim (bridge table)")
    df = df.copy()

    dupes = df.duplicated(subset=["job_id", "skill_id"]).sum()
    df = df.drop_duplicates(subset=["job_id", "skill_id"])
    print(f"  Dropped {dupes} duplicate (job_id, skill_id) pairs")

    if job_postings is not None:
        orphan_jobs = (~df["job_id"].isin(job_postings["job_id"])).sum()
        print(f"  Orphan job_id references (no matching posting): {orphan_jobs:,}")

    if skills_dim is not None:
        orphan_skills = (~df["skill_id"].isin(skills_dim["skill_id"])).sum()
        print(f"  Orphan skill_id references (no matching skill): {orphan_skills:,}")

    return df


# ----------------------------------------------------------------------
# 6. RUN PIPELINE
# ----------------------------------------------------------------------
if company_dim is not None:
    company_dim_clean = clean_company_dim(company_dim)
    company_dim_clean.to_csv(os.path.join(OUT_DIR, "company_dim_cleaned.csv"), index=False)

if skills_dim is not None:
    skills_dim_clean = clean_skills_dim(skills_dim)
    skills_dim_clean.to_csv(os.path.join(OUT_DIR, "skills_dim_cleaned.csv"), index=False)

if job_postings is not None:
    job_postings_clean = clean_job_postings(job_postings)
    job_postings_clean.to_csv(os.path.join(OUT_DIR, "job_postings_cleaned.csv"), index=False)

if skills_job is not None:
    skills_job_clean = clean_skills_job(
        skills_job,
        job_postings_clean if job_postings is not None else None,
        skills_dim_clean if skills_dim is not None else None,
    )
    skills_job_clean.to_csv(os.path.join(OUT_DIR, "skills_job_cleaned.csv"), index=False)


# ----------------------------------------------------------------------
# 7. BUILD MASTER ANALYSIS TABLE (long format: one row per posting-skill)
# ----------------------------------------------------------------------
if job_postings is not None and skills_job is not None and skills_dim is not None and company_dim is not None:
    log("STEP 7: BUILDING MASTER ANALYSIS TABLE")

    master = (
        job_postings_clean
        .merge(company_dim_clean[["company_id", "name", "has_website"]],
                on="company_id", how="left")
        .rename(columns={"name": "company_name"})
        .merge(skills_job_clean, on="job_id", how="left")
        .merge(skills_dim_clean[["skill_id", "skills", "type", "skill_era"]],
                on="skill_id", how="left")
    )

    # skill count per posting — a genuinely useful derived feature
    skill_counts = skills_job_clean.groupby("job_id").size().rename("skill_count_per_posting")
    job_postings_clean = job_postings_clean.merge(skill_counts, on="job_id", how="left")
    job_postings_clean["skill_count_per_posting"] = job_postings_clean["skill_count_per_posting"].fillna(0)
    job_postings_clean.to_csv(os.path.join(OUT_DIR, "job_postings_cleaned.csv"), index=False)

    master.to_csv(os.path.join(OUT_DIR, "master_analysis_table.csv"), index=False)
    print(f"  Master table shape: {master.shape}")

log("DONE — cleaned files written to: " + OUT_DIR)



STEP 1: LOADING RAW TABLES


C:\Users\Manas\AppData\Local\Temp\ipykernel_4728\1777338072.py:44: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(path, **kwargs)


  [OK] job_postings_fact.csv: 787,686 rows x 16 cols
  [OK] company_dim.csv: 140,033 rows x 5 cols
  [OK] skills_dim.csv: 259 rows x 3 cols
  [OK] skills_job_dim.csv: 3,669,604 rows x 2 cols

STEP 2: CLEANING company_dim
  Dropped 3 rows with null company name
  Dropped 'thumbnail' column (no analytical value)
  Flagged 18477 rows sharing a duplicate company name
  Final shape: (140030, 6)

STEP 3: CLEANING skills_dim
  Found 7 duplicate skill names — dropping duplicates, keeping first
  Final shape: (252, 4)

STEP 4: CLEANING job_postings_fact
  Dropped 0 duplicate job_id rows
  Salary type breakdown:
salary_type
missing             754987
yearly               22034
hourly_converted     10665
  Flagged 476 salary outliers (IQR bounds: -14,045 - 241,227)
  Seniority breakdown:
seniority_level
Mid/Unspecified       566038
Senior                147928
Lead                   29797
Principal/Director     23005
Junior/Entry           20918
  Rows before: 787,686 -> after: 787,686

STEP 5: C